In [1]:
# 1. Mount Drive and go to the project folder
from google.colab import drive
drive.mount('/content/drive')
%mkdir -p /content/drive/MyDrive/research-engine
%cd /content/drive/MyDrive/research-engine

Mounted at /content/drive
/content/drive/MyDrive/research-engine


In [2]:
# 2. Install deps (Colab already has torch and sentence-transformers)
!pip install -q datasets rank_bm25

In [3]:
# 1. Search the whole Drive for any chunks file (also shows your project folder's full path)
!find /content/drive/MyDrive -iname "*chunk*" -not -path "*/.Trash*" 2>/dev/null

# 2. See where build_chunks.py writes its output
!find /content/drive/MyDrive -name build_chunks.py -not -path "*/.Trash*" -exec grep -nE "to_parquet|to_json|\.jsonl|open\(|save" {} \;

/content/drive/MyDrive/research-engine/research-engine/ingest/build_chunks.py
/content/drive/MyDrive/research-engine/ingest/build_chunks.py
/content/drive/MyDrive/research-engine/data/processed/chunks.parquet
10:    qasper_questions.jsonl   QASPER questions with gold chunk IDs (seed for the eval set)
170:    pd.DataFrame(papers).to_parquet(OUT_DIR / "papers.parquet", index=False)
171:    chunks_df.to_parquet(OUT_DIR / "chunks.parquet", index=False)
172:    with open(OUT_DIR / "qasper_questions.jsonl", "w", encoding="utf-8") as fh:
200:    pd.DataFrame(papers).to_parquet(OUT_DIR / "papers.parquet", index=False)
201:    chunks_df.to_parquet(OUT_DIR / "chunks.parquet", index=False)
202:    with open(OUT_DIR / "qasper_questions.jsonl", "w", encoding="utf-8") as fh:


In [4]:
import pandas as pd, json

df = pd.read_parquet("data/processed/chunks.parquet")
print(df.shape)
print(df.dtypes)
print({k: (str(v)[:120]) for k, v in df.iloc[0].items()})

with open("data/processed/qasper_questions.jsonl") as f:
    rows = [json.loads(l) for l in f]
print(len(rows))
print({k: (str(v)[:150]) for k, v in rows[0].items()})

(28541, 9)
chunk_id        object
paper_id        object
title           object
section         object
section_type    object
section_pos      int64
chunk_pos        int64
n_words          int64
text            object
dtype: object
{'chunk_id': '1912.01214::000', 'paper_id': '1912.01214', 'title': 'Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation', 'section': 'Abstract', 'section_type': 'abstract', 'section_pos': '0', 'chunk_pos': '0', 'n_words': '165', 'text': '[Abstract] Transfer learning between different language pairs has shown its effectiveness for Neural Machine Translation'}
4294
{'question_id': 'b6f15fb6279b82e34a5bf4828b7b5ddabfdf1d54', 'question': 'which multilingual approaches do they compare with?', 'paper_id': '1912.01214', 'paper_title': 'Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation', 'split': 'dev', 'type': 'single_paper', 'gold_chunk_ids': "['1912.01214::012']", 'reference_answers': "['BIBREF19; BI

In [5]:
%cd /content/drive/MyDrive/research-engine
!pip install -q rank_bm25
!python day3_hybrid_retrieval.py --limit 200            # quick check, ~2–3 min (first run embeds all chunks)
!python day3_hybrid_retrieval.py --scope paper
!python day3_hybrid_retrieval.py --scope corpus

/content/drive/MyDrive/research-engine
28541 chunks | 1585 papers | 200 questions with gold evidence (split=dev)
modules.json: 100% 349/349 [00:00<00:00, 1.88MB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 595kB/s]
README.md: 100% 94.8k/94.8k [00:00<00:00, 75.2MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 273kB/s]
config.json: 100% 743/743 [00:00<00:00, 3.24MB/s]

model.safetensors: downloading bytes:  57% 76.2M/133M [00:00<00:00, 133MB/s, 5.11MB/s  ]
model.safetensors: reconstructing file:  80% 107M/133M [00:00<00:00, 108MB/s]
model.safetensors: downloading bytes: 100% 86.7M/86.7M [00:01<00:00, 78.2MB/s, 8.28MB/s  ]
model.safetensors: reconstructing file: 100% 133M/133M [00:01<00:00, 120MB/s, 12.9MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4957.19it/s]
tokenizer_config.json: 100% 366/366 [00:00<00:00, 1.34MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 25.6MB/s]
tokenizer.json: 100% 711k/711k [00:00<00:00, 53.2MB/s]
special_tokens_map.json: 10